In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+------------------------+
|catalog                 |
+------------------------+
|healthcare_medallion_dbw|
|samples                 |
|system                  |
+------------------------+



In [0]:
CATALOG = "healthcare_medallion_dbw"

BRONZE = "bronze"
SILVER = "silver"
GOLD = "gold"
OPS = "ops"

print("Catalog:", CATALOG)

Catalog: healthcare_medallion_dbw


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{OPS}")

print("Schemas created successfully")

Schemas created successfully


In [0]:
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|ops               |
|silver            |
+------------------+



In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{OPS}.metadata_config (
    source_id STRING,
    source_name STRING,
    file_path STRING,
    file_format STRING,
    target_table STRING,
    primary_key STRING,
    active_flag STRING,
    load_type STRING,
    last_load_timestamp TIMESTAMP,
    expected_schema_version STRING,
    source_system_owner STRING,
    data_classification STRING,
    row_count_threshold BIGINT,
    max_null_pct DOUBLE,
    notification_email STRING,
    retry_count INT,
    quarantine_path STRING,
    partition_column STRING,
    sla_cutoff_time STRING,
    dependency_source_ids STRING
)
USING DELTA
""")

print("metadata_config created")

metadata_config created


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{OPS}.pipeline_audit_log (
    audit_id STRING,
    batch_id STRING,
    source_name STRING,
    layer STRING,
    pipeline_start_time TIMESTAMP,
    pipeline_end_time TIMESTAMP,
    rows_read BIGINT,
    rows_written BIGINT,
    rows_rejected BIGINT,
    status STRING,
    error_message STRING,
    triggered_by STRING,
    created_at TIMESTAMP,
    pipeline_duration_secs BIGINT,
    notebook_name STRING,
    cluster_id STRING,
    spark_app_id STRING,
    rows_quarantined BIGINT,
    dq_score_avg DOUBLE,
    schema_version STRING,
    environment STRING,
    retry_attempt INT,
    data_classification STRING,
    sla_met BOOLEAN,
    downstream_notified BOOLEAN
)
USING DELTA
""")

print("pipeline_audit_log created")

pipeline_audit_log created


In [0]:
spark.sql(
    f"SHOW TABLES IN {CATALOG}.{OPS}"
).show(truncate=False)

+--------+------------------+-----------+
|database|tableName         |isTemporary|
+--------+------------------+-----------+
|ops     |metadata_config   |false      |
|ops     |pipeline_audit_log|false      |
+--------+------------------+-----------+



In [0]:
for schema in ["bronze", "silver", "gold", "ops"]:
    print(f"\n--- {schema.upper()} ---")
    spark.sql(
        f"SHOW TABLES IN {CATALOG}.{schema}"
    ).show(truncate=False)


--- BRONZE ---
+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+


--- SILVER ---
+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+


--- GOLD ---
+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+


--- OPS ---
+--------+------------------+-----------+
|database|tableName         |isTemporary|
+--------+------------------+-----------+
|ops     |metadata_config   |false      |
|ops     |pipeline_audit_log|false      |
+--------+------------------+-----------+



In [0]:
landing_path = "abfss://landing@tanvihealthstore2608.dfs.core.windows.net/"

display(dbutils.fs.ls(landing_path))

path,name,size,modificationTime
abfss://landing@tanvihealthstore2608.dfs.core.windows.net/appointments/,appointments/,0,1786369349000
abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/,billings/,0,1786369417000
abfss://landing@tanvihealthstore2608.dfs.core.windows.net/doctors/,doctors/,0,1786368999000
abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/,patients/,0,1786368948000
abfss://landing@tanvihealthstore2608.dfs.core.windows.net/treatments/,treatments/,0,1786369383000


In [0]:
patients_test = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(
        "abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv"
    )
)

display(patients_test)

patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email
P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com
P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com
P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com
P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com
P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com
P006,Linda,Jones,M,1963-06-16,7561777264,321 Maple Dr,2022-10-02,HealthIndia,INS613758,linda.jones@mail.com
P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com
P008,David,Davis,F,1976-07-05,7090558393,456 Oak Ave,2021-05-25,WellnessCorp,INS545101,david.davis@mail.com
P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com
P010,Michael,Taylor,M,2001-10-13,7081396733,123 Elm St,2022-08-24,WellnessCorp,INS866577,michael.taylor@mail.com
